In [1]:
%pip install supabase python-dotenv pandas numpy scipy scikit-learn PySastrawi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# =========================================================
# CELL 1 - IMPORT LIBRARY + KONEKSI SUPABASE
# =========================================================
import os
import re
import json
import sys
import scipy.sparse as sp
import pandas as pd
import numpy as np

from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from dotenv import load_dotenv
from supabase import create_client
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.metrics.pairwise import cosine_similarity

BACKEND_DIR = next(
    path for path in (Path.cwd(), Path.cwd().parent, Path.cwd() / "backend")
    if (path / "src" / "preprocessing").is_dir()
)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from src.preprocessing.stopwords import get_stopwords

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY") or os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

SOURCE_TABLE = "cleaned_papers_results"
EVAL_TABLE = "evaluation_precision_at_k"

base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
tfidf_dir = os.path.join(base_dir, "data", "tfidf")
eval_dir = os.path.join(base_dir, "data", "evaluation")

os.makedirs(eval_dir, exist_ok=True)

print("✅ Import dan koneksi Supabase berhasil")

✅ Import dan koneksi Supabase berhasil


In [3]:
# =========================================================
# CELL 2 - LOAD FILE VSM HASIL TF-IDF
# =========================================================
tfidf_matrix = sp.load_npz(os.path.join(tfidf_dir, "tfidf_matrix.npz"))

with open(os.path.join(tfidf_dir, "tfidf_terms.json"), "r", encoding="utf-8") as file:
    terms = json.load(file)

with open(os.path.join(tfidf_dir, "tfidf_doc_ids.json"), "r", encoding="utf-8") as file:
    doc_ids = json.load(file)

with open(os.path.join(tfidf_dir, "idf_scores.json"), "r", encoding="utf-8") as file:
    idf_scores = json.load(file)

tfidf_documents = pd.read_csv(os.path.join(tfidf_dir, "tfidf_documents.csv"))

doc_ids = [int(doc_id) for doc_id in doc_ids]

print("✅ File VSM berhasil dimuat")
print("Matrix TF-IDF:", tfidf_matrix.shape)
print("Jumlah terms:", len(terms))
print("Jumlah doc_ids:", len(doc_ids))

✅ File VSM berhasil dimuat
Matrix TF-IDF: (200, 1732)
Jumlah terms: 1732
Jumlah doc_ids: 200


In [4]:
# =========================================================
# CELL 3 - LOAD METADATA ARTIKEL DARI SUPABASE
# =========================================================
all_data = []
batch_size = 1000
offset = 0

selected_columns = """
id,
title,
abstract,
authors,
year,
source,
category,
pdf_url,
url,
scrape_status
"""

while True:
    response = (
        supabase.table(SOURCE_TABLE)
        .select(selected_columns)
        .range(offset, offset + batch_size - 1)
        .execute()
    )

    batch = response.data or []

    if not batch:
        break

    all_data.extend(batch)

    if len(batch) < batch_size:
        break

    offset += batch_size

metadata_df = pd.DataFrame(all_data)

if metadata_df.empty:
    raise ValueError("❌ Data cleaned_papers_results kosong.")

metadata_df["id"] = metadata_df["id"].astype("int64")

doc_index = pd.DataFrame({"id": doc_ids})
doc_index = doc_index.merge(metadata_df, on="id", how="left")
doc_index = doc_index.merge(
    tfidf_documents[["id", "document_text"]],
    on="id",
    how="left"
)

for col in ["title", "abstract", "authors", "source", "category", "pdf_url", "url", "scrape_status", "document_text"]:
    if col in doc_index.columns:
        doc_index[col] = doc_index[col].fillna("")

if tfidf_matrix.shape[0] != len(doc_index):
    raise ValueError("❌ Jumlah matrix TF-IDF tidak sama dengan jumlah dokumen.")

print("✅ Metadata artikel siap")
print("Jumlah dokumen:", len(doc_index))
doc_index.head(3)

✅ Metadata artikel siap
Jumlah dokumen: 200


,id,title,abstract,authors,year,source,category,pdf_url,url,scrape_status,document_text
0,1,Penerapan machine learning dalam prediksi ting...,… menjabarkan implementasi machine learning un...,"RG Wardhana, G Wang…",2023,Journal of Information …,machine learning,https://jurnal.amikom.ac.id/index.php/joism/ar...,https://jurnal.amikom.ac.id/index.php/joism/ar...,pdf_downloaded,terap machine learning prediksi tingkat kasus ...
1,2,Tinjauan Pustaka Sistematis: Penerapan Metode ...,… Machine Learning dapat mempelajari pola data...,"IM Faiza, W Andriani",2022,Jurnal Minfo Polgan,machine learning,https://jurnal.polgan.ac.id/index.php/jmp/arti...,https://jurnal.polgan.ac.id/index.php/jmp/arti...,pdf_downloaded,tinjau pustaka sistematis terap metode machine...
2,3,"Penerapan Machine Learning, Deep Learning, Dan...","… the application of machine learning, deep le...","S Prasetyo, T Dewayanto",2024,Diponegoro Journal of Accounting,machine learning,https://ejournal3.undip.ac.id/index.php/accoun...,https://ejournal3.undip.ac.id/index.php/accoun...,pdf_downloaded,terap machine learning deep learning data mini...


In [5]:
# =========================================================
# CELL 4 - STOPWORDS + STEMMER
# =========================================================
stop_words = get_stopwords()

stemmer = StemmerFactory().create_stemmer()

print("✅ Stopwords dan stemmer siap")

✅ Stopwords dan stemmer siap


In [6]:
# =========================================================
# CELL 5 - FUNGSI QUERY, COSINE, DAN SEARCH
# =========================================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_query(query):
    cleaned = clean_text(query)
    tokens = cleaned.split()
    tokens = [token for token in tokens if token not in stop_words and len(token) > 1]
    tokens = [stemmer.stem(token) for token in tokens]
    return tokens

def build_query_vector(query_tokens):
    term_to_index = {term: index for index, term in enumerate(terms)}
    query_vector = np.zeros(len(terms), dtype=float)

    if not query_tokens:
        return query_vector.reshape(1, -1)

    total_terms = len(query_tokens)
    term_counts = Counter(query_tokens)

    for term, count in term_counts.items():
        if term in term_to_index:
            tf = count / total_terms
            idf = float(idf_scores.get(term, 0))
            query_vector[term_to_index[term]] = tf * idf

    return query_vector.reshape(1, -1)

def count_occurrence(document_text, query_tokens):
    document_tokens = str(document_text).split()
    return sum(document_tokens.count(term) for term in query_tokens)

def interpret_similarity(score, threshold=0.3):
    if score > 0.5:
        return "Relevan Tinggi"
    if score > threshold:
        return "Relevan Sedang"
    return "Rendah"

def search(query, top_k=20, min_occurrence=0, threshold=0.3):
    query_tokens = preprocess_query(query)

    if not query_tokens:
        return pd.DataFrame()

    query_vector = build_query_vector(query_tokens)
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    results = doc_index.copy()
    results["similarity_score"] = scores
    results["occurrence"] = results["document_text"].apply(
        lambda text: count_occurrence(text, query_tokens)
    )

    results = results[results["similarity_score"] > 0].copy()

    if min_occurrence > 0:
        results = results[results["occurrence"] >= min_occurrence].copy()

    results = results.sort_values("similarity_score", ascending=False)
    results = results.head(top_k).reset_index(drop=True)

    results["rank"] = range(1, len(results) + 1)
    results["threshold_relevan"] = (results["similarity_score"] > threshold).astype(int)
    results["interpretation"] = results["similarity_score"].apply(
        lambda score: interpret_similarity(score, threshold)
    )

    return results

print("✅ Fungsi search evaluasi siap")

✅ Fungsi search evaluasi siap


In [7]:
# =========================================================
# CELL 6 - QUERY UJI EVALUASI
# K = 5, 10, 20 sesuai proposal
# Query dibuat bilingual karena dataset berisi artikel Indonesia dan Inggris.
# =========================================================
queries_eval = [
    {"query": "machine learning", "kategori": "Machine Learning"},
    {"query": "pembelajaran mesin", "kategori": "Machine Learning"},
    {"query": "data mining", "kategori": "Machine Learning"},

    {"query": "web application", "kategori": "Web Application"},
    {"query": "aplikasi web", "kategori": "Web Application"},
    {"query": "sistem informasi berbasis web", "kategori": "Web Application"},

    {"query": "cyber security", "kategori": "Cyber Security"},
    {"query": "keamanan siber", "kategori": "Cyber Security"},
    {"query": "keamanan jaringan", "kategori": "Cyber Security"},

    {"query": "mobile application", "kategori": "Mobile Application"},
    {"query": "aplikasi mobile", "kategori": "Mobile Application"},
    {"query": "aplikasi android", "kategori": "Mobile Application"},
]

K_VALUES = [5, 10, 20]
MAX_K = max(K_VALUES)
THRESHOLD = 0.3

print("? Query evaluasi bilingual siap")


? Query evaluasi bilingual siap


In [8]:
# =========================================================
# CELL 7 - JALANKAN SEARCH UNTUK SEMUA QUERY
# =========================================================
all_results = {}
kemunculan_all = defaultdict(int)
kemunculan_kat = defaultdict(lambda: defaultdict(int))

for item in queries_eval:
    query = item["query"]
    kategori = item["kategori"]

    results = search(
        query=query,
        top_k=MAX_K,
        min_occurrence=0,
        threshold=THRESHOLD
    )

    all_results[query] = results

    for _, row in results.iterrows():
        kemunculan_all[row["title"]] += 1
        kemunculan_kat[kategori][row["title"]] += 1

print("✅ Semua query selesai diproses")

✅ Semua query selesai diproses


In [ ]:
# =========================================================
# CELL 8 - SUSUN HASIL PENCARIAN UNTUK DICOCOKKAN DENGAN LABEL EVALUATOR
# Data ini hanya dipakai di memori notebook, tidak disimpan sebagai file terpisah.
# Ground truth final berasal dari ground_truth_evaluation_3_evaluator.csv.
# =========================================================
rows = []

for item in queries_eval:
    query = item["query"]
    kategori_query = item["kategori"]
    results = all_results[query]

    for _, row in results.iterrows():
        rows.append({
            "query": query,
            "kategori_query": kategori_query,
            "rank": int(row["rank"]),
            "article_id": int(row["id"]),
            "article_category": row["category"],
            "title": row["title"],
            "abstract": row["abstract"],
            "similarity_score": float(row["similarity_score"]),
            "threshold_relevan": int(row["threshold_relevan"]),
        })

hasil_pencarian_eval_df = pd.DataFrame(rows)

print("[OK] Hasil pencarian evaluasi siap dicocokkan dengan Label Final evaluator")
print("Jumlah data:", len(hasil_pencarian_eval_df))
hasil_pencarian_eval_df.head(10)





In [ ]:
# =========================================================
# CELL 9 - LOAD GROUND TRUTH FINAL DARI 3 EVALUATOR
# Label Final evaluator dipakai untuk Precision@K.
# threshold_relevan hanya disimpan sebagai indikator awal, bukan label final.
# =========================================================
evaluator_path = os.path.join(eval_dir, "ground_truth_evaluation_3_evaluator.csv")
evaluator_df = pd.read_csv(evaluator_path)

evaluator_df["query_key"] = evaluator_df["Query"].astype(str).str.lower().str.strip()
evaluator_df["article_id"] = pd.to_numeric(evaluator_df["Article ID"], errors="coerce").astype("Int64")
evaluator_df["label_final"] = pd.to_numeric(
    evaluator_df["Label Final"], errors="coerce"
).fillna(0).astype(int)

labels_df = evaluator_df[["query_key", "article_id", "label_final"]].copy()

gt_df = hasil_pencarian_eval_df.copy()
gt_df["query_key"] = gt_df["query"].astype(str).str.lower().str.strip()
gt_df["article_id"] = pd.to_numeric(gt_df["article_id"], errors="coerce").astype("Int64")

gt_df = gt_df.merge(labels_df, on=["query_key", "article_id"], how="left")
gt_df["relevan"] = gt_df["label_final"].astype(int)

gt_df["evaluator_1"] = pd.to_numeric(gt_df["Evaluator 1"], errors="coerce").fillna(0).astype(int)
gt_df["evaluator_2"] = pd.to_numeric(gt_df["Evaluator 2"], errors="coerce").fillna(0).astype(int)
gt_df["evaluator_3"] = pd.to_numeric(gt_df["Evaluator 3"], errors="coerce").fillna(0).astype(int)
gt_df["status_final"] = gt_df.get("Status Final", "").fillna("")
gt_df["catatan_evaluator"] = gt_df.get("Catatan Evaluator", "").fillna("")

invalid = set(gt_df["relevan"].unique()) - {0, 1}
if invalid:
    raise ValueError(f"Label Final harus 0/1. Ditemukan: {invalid}")


gt_df["query"] = gt_df["query"].astype(str).str.lower().str.strip()
gt_df["article_id"] = gt_df["article_id"].astype("int64")
gt_df["relevan"] = gt_df["relevan"].astype(int)

print("[OK] Ground truth final dari 3 evaluator siap dipakai di memori notebook")
print("File sumber:", evaluator_path)
print("Jumlah data:", len(gt_df))
print("Distribusi label final:")
print(gt_df["relevan"].value_counts().sort_index())
gt_df.head(10)






In [11]:
# =========================================================
# CELL 10 - TABEL 1: HASIL PENCARIAN PER QUERY + LABEL
# =========================================================
tabel1_rows = []

for item in queries_eval:
    query = item["query"]
    kategori_query = item["kategori"]
    results = all_results[query].copy()

    sub_gt = gt_df[gt_df["query"] == query][["article_id", "relevan"]].copy()
    sub_gt = sub_gt.rename(columns={"article_id": "id"})

    merged = results.merge(sub_gt, on="id", how="left")
    merged["relevan"] = merged["relevan"].fillna(0).astype(int)

    for _, row in merged.iterrows():
        tabel1_rows.append({
            "Query": query,
            "Kategori Query": kategori_query,
            "Rank": int(row["rank"]),
            "Article ID": int(row["id"]),
            "Kategori Artikel": row["category"],
            "Judul": row["title"],
            "Penulis": row["authors"],
            "Tahun": row["year"],
            "Source": row["source"],
            "Similarity Score": round(float(row["similarity_score"]), 6),
            "Threshold Relevan": "Ya" if row["threshold_relevan"] == 1 else "Tidak",
            "Relevan Human Judgment": "Ya" if row["relevan"] == 1 else "Tidak",
            "Occurrence": int(row["occurrence"]),
            "Interpretasi": row["interpretation"]
        })

tabel1_df = pd.DataFrame(tabel1_rows)

tabel1_path = os.path.join(eval_dir, "tabel1_hasil_pencarian.csv")
tabel1_df.to_csv(tabel1_path, index=False)

print("✅ Tabel 1 tersimpan:", tabel1_path)
tabel1_df.head(10)

✅ Tabel 1 tersimpan: d:\Tugas Akhir\paperci_artikel\backend\data\evaluation\tabel1_hasil_pencarian.csv


,Query,Kategori Query,Rank,Article ID,Kategori Artikel,Judul,Penulis,Tahun,Source,Similarity Score,Threshold Relevan,Relevan Human Judgment,Occurrence,Interpretasi
0,machine learning,Machine Learning,1,3,machine learning,"Penerapan Machine Learning, Deep Learning, Dan...","S Prasetyo, T Dewayanto",2024,Diponegoro Journal of Accounting,0.474999,Ya,Ya,12,Relevan Sedang
1,machine learning,Machine Learning,2,30,machine learning,Machine learning and deep learning: C. Janiesc...,"C Janiesch, P Zschech, K Heinrich",2021,Electronic markets,0.469477,Ya,Ya,12,Relevan Sedang
2,machine learning,Machine Learning,3,13,machine learning,Penerapan Machine Learning Untuk Mengategorika...,"H Hendri, L Hoki, V Agusman, D Aryanto",2021,Jurnal TIMES,0.445702,Ya,Ya,9,Relevan Sedang
3,machine learning,Machine Learning,4,40,machine learning,An overview of machine learning classification...,"AFAH Alnuaimi, THK Albaldawi",2024,BIO Web of Conferences,0.431999,Ya,Ya,10,Relevan Sedang
4,machine learning,Machine Learning,5,1,machine learning,Penerapan machine learning dalam prediksi ting...,"RG Wardhana, G Wang…",2023,Journal of Information …,0.405156,Ya,Ya,8,Relevan Sedang
5,machine learning,Machine Learning,6,23,machine learning,Penerapan Metode Machine Learning dalam Mengid...,"VA Saputra, SA Arnomo",2024,Computer Based Information …,0.384833,Ya,Ya,8,Relevan Sedang
6,machine learning,Machine Learning,7,2,machine learning,Tinjauan Pustaka Sistematis: Penerapan Metode ...,"IM Faiza, W Andriani",2022,Jurnal Minfo Polgan,0.382231,Ya,Ya,8,Relevan Sedang
7,machine learning,Machine Learning,8,22,machine learning,Pemanfaatan machine learning di bidang kesehatan,"I Akbar, F Supriadi, DI Junaedi",2025,JATI (Jurnal Mahasiswa Teknik …,0.371533,Ya,Ya,8,Relevan Sedang
8,machine learning,Machine Learning,9,21,machine learning,Penerapan Teknologi Machine Learning dalam Det...,F Putra,2024,Jurnal Kolaborasi Sains dan Ilmu Terapan,0.367296,Ya,Ya,8,Relevan Sedang
9,machine learning,Machine Learning,10,38,machine learning,A survey on dataset quality in machine learning,"Y Gong, G Liu, Y Xue, R Li, L Meng",2023,Information and Software Technology,0.360741,Ya,Ya,8,Relevan Sedang


In [12]:
# =========================================================
# CELL 11 - TABEL 2: PRECISION@K
# Precision@K = jumlah relevan di Top K / K
# =========================================================
eval_rows = []

for item in queries_eval:
    query = item["query"]
    kategori_query = item["kategori"]
    results = all_results[query].copy()

    sub_gt = gt_df[gt_df["query"] == query][["article_id", "relevan"]].copy()
    sub_gt = sub_gt.rename(columns={"article_id": "id"})

    merged = results.merge(sub_gt, on="id", how="left")
    merged["relevan"] = merged["relevan"].fillna(0).astype(int)

    row_eval = {
        "Query": query,
        "Kategori": kategori_query,
        "Retrieved": int(len(merged))
    }

    for k in K_VALUES:
        top_k = merged.head(k)
        relevant_k = int(top_k["relevan"].sum())
        precision_k = relevant_k / k

        row_eval[f"Relevan@{k}"] = relevant_k
        row_eval[f"P@{k}"] = round(precision_k, 4)

    eval_rows.append(row_eval)

eval_df = pd.DataFrame(eval_rows)

eval_path = os.path.join(eval_dir, "tabel2_precision_at_k.csv")
eval_df.to_csv(eval_path, index=False)

print("✅ Tabel 2 Precision@K tersimpan:", eval_path)
eval_df

✅ Tabel 2 Precision@K tersimpan: d:\Tugas Akhir\paperci_artikel\backend\data\evaluation\tabel2_precision_at_k.csv


,Query,Kategori,Retrieved,Relevan@5,P@5,Relevan@10,P@10,Relevan@20,P@20
0,machine learning,Machine Learning,20,5,1.0,10,1.0,20,1.00
1,pembelajaran mesin,Machine Learning,5,2,0.4,2,0.2,2,0.10
2,data mining,Machine Learning,20,0,0.0,0,0.0,0,0.00
3,web application,Web Application,20,4,0.8,4,0.4,4,0.20
4,aplikasi web,Web Application,20,5,1.0,6,0.6,6,0.30
5,sistem informasi berbasis web,Web Application,20,2,0.4,2,0.2,2,0.10
6,cyber security,Cyber Security,20,5,1.0,10,1.0,19,0.95
7,keamanan siber,Cyber Security,20,5,1.0,10,1.0,20,1.00
8,keamanan jaringan,Cyber Security,20,1,0.2,1,0.1,1,0.05
9,mobile application,Mobile Application,20,1,0.2,1,0.1,1,0.05


In [13]:
# =========================================================
# CELL 12 - RATA-RATA PRECISION
# =========================================================
summary_rows = []

for k in K_VALUES:
    summary_rows.append({
        "Metric": f"Mean P@{k}",
        "Value": round(eval_df[f"P@{k}"].mean(), 4),
        "Percentage": round(eval_df[f"P@{k}"].mean() * 100, 2)
    })

summary_df = pd.DataFrame(summary_rows)

kat_avg_rows = []

for kategori, group in eval_df.groupby("Kategori"):
    row = {"Kategori": kategori}
    for k in K_VALUES:
        row[f"Rata-rata P@{k}"] = round(group[f"P@{k}"].mean(), 4)
    kat_avg_rows.append(row)

kat_avg_df = pd.DataFrame(kat_avg_rows)

summary_path = os.path.join(eval_dir, "tabel2_rata_rata_keseluruhan.csv")
kat_avg_path = os.path.join(eval_dir, "tabel2_rata_rata_per_kategori.csv")

summary_df.to_csv(summary_path, index=False)
kat_avg_df.to_csv(kat_avg_path, index=False)

print("✅ Rata-rata keseluruhan:")
print(summary_df.to_string(index=False))

print("\n✅ Rata-rata per kategori:")
kat_avg_df

✅ Rata-rata keseluruhan:
   Metric  Value  Percentage
 Mean P@5 0.6000       60.00
Mean P@10 0.4750       47.50
Mean P@20 0.3667       36.67

✅ Rata-rata per kategori:


,Kategori,Rata-rata P@5,Rata-rata P@10,Rata-rata P@20
0,Cyber Security,0.7333,0.7,0.6667
1,Machine Learning,0.4667,0.4,0.3667
2,Mobile Application,0.4667,0.4,0.2333
3,Web Application,0.7333,0.4,0.2000


In [ ]:
# =========================================================
# CELL 13 - CATATAN OUTPUT TAMBAHAN
# Tabel kemunculan artikel lintas query tidak disimpan karena tidak digunakan
# pada pembahasan Precision@K final.
# =========================================================
print("[OK][OK] Output tabel kemunculan tidak dibuat. Fokus evaluasi: ground truth evaluator dan Precision@K.")




In [ ]:
# =========================================================
# CELL 14 - SIMPAN OUTPUT FINAL EVALUASI PRECISION@K
# File yang disimpan hanya file final yang dipakai untuk Bab IV/lampiran.
# =========================================================
os.makedirs(eval_dir, exist_ok=True)

tabel1_df.to_csv(
    os.path.join(eval_dir, "tabel1_hasil_pencarian.csv"),
    index=False
)

eval_df.to_csv(
    os.path.join(eval_dir, "tabel2_precision_at_k.csv"),
    index=False
)

summary_df.to_csv(
    os.path.join(eval_dir, "tabel2_rata_rata_keseluruhan.csv"),
    index=False
)

kat_avg_df.to_csv(
    os.path.join(eval_dir, "tabel2_rata_rata_per_kategori.csv"),
    index=False
)

print("\n[OK] File final evaluasi tersimpan di data/evaluation/")
print(" - ground_truth_evaluation_3_evaluator.csv")
print(" - tabel1_hasil_pencarian.csv")
print(" - tabel2_precision_at_k.csv")
print(" - tabel2_rata_rata_keseluruhan.csv")
print(" - tabel2_rata_rata_per_kategori.csv")





In [16]:
# =========================================================
# CELL 15 - HELPER UPSERT SUPABASE
# =========================================================
def to_records_safe(dataframe):
    return dataframe.where(pd.notnull(dataframe), None).to_dict(orient="records")

def upsert_batches(table_name, records, on_conflict, batch_size=500):
    total = len(records)

    if total == 0:
        print(f"⚠️ Tidak ada data untuk disimpan ke {table_name}")
        return

    for start in range(0, total, batch_size):
        batch = records[start:start + batch_size]

        supabase.table(table_name).upsert(
            batch,
            on_conflict=on_conflict
        ).execute()

        print(f"  → Batch {start // batch_size + 1}: {len(batch)} data")

    print(f"✅ Upsert {total} data ke {table_name}")

In [17]:
# =========================================================
# CELL 16 - SIMPAN EVALUATION KE SUPABASE
# table: evaluation_precision_at_k
# K = 5, 10, 20 sesuai proposal
# =========================================================
ts = datetime.now(timezone.utc).isoformat()

rows_eval_db = []

for _, row in eval_df.iterrows():
    query = row["Query"]

    for k in K_VALUES:
        rows_eval_db.append({
            "compared_text": query,
            "k": int(k),
            "retrieved_count": int(row["Retrieved"]),
            "relevant_retrieved": int(row[f"Relevan@{k}"]),
            "precision_at_k": float(row[f"P@{k}"]),
            "updated_at": ts
        })

upsert_batches(
    table_name=EVAL_TABLE,
    records=rows_eval_db,
    on_conflict="compared_text,k",
    batch_size=500
)

  → Batch 1: 36 data
✅ Upsert 36 data ke evaluation_precision_at_k


In [18]:
# =========================================================
# CELL 17 - VALIDASI SUPABASE
# =========================================================
check = (
    supabase.table(EVAL_TABLE)
    .select("compared_text,k,precision_at_k", count="exact")
    .limit(40)
    .execute()
)

print("✅ Total row evaluation_precision_at_k:", check.count)
check.data

✅ Total row evaluation_precision_at_k: 54


[{'compared_text': 'android application', 'k': 5, 'precision_at_k': 0},
 {'compared_text': 'android application', 'k': 10, 'precision_at_k': 0},
 {'compared_text': 'android application', 'k': 20, 'precision_at_k': 0},
 {'compared_text': 'machine learning', 'k': 5, 'precision_at_k': 1},
 {'compared_text': 'machine learning', 'k': 10, 'precision_at_k': 1},
 {'compared_text': 'machine learning', 'k': 20, 'precision_at_k': 1},
 {'compared_text': 'pembelajaran mesin', 'k': 5, 'precision_at_k': 0.4},
 {'compared_text': 'deep learning', 'k': 5, 'precision_at_k': 0.8},
 {'compared_text': 'deep learning', 'k': 10, 'precision_at_k': 0.4},
 {'compared_text': 'deep learning', 'k': 20, 'precision_at_k': 0.2},
 {'compared_text': 'website application', 'k': 5, 'precision_at_k': 0},
 {'compared_text': 'website application', 'k': 10, 'precision_at_k': 0},
 {'compared_text': 'website application', 'k': 20, 'precision_at_k': 0},
 {'compared_text': 'web system', 'k': 5, 'precision_at_k': 0},
 {'compared_t